# Semana 5 - Pandas vs Polars

## Comparativa básica entre Pandas y Polars

Preparación: importamos ambas librerías para tenerlas disponibles en el resto del cuaderno.
- Propósito: cargar los módulos y comprobar versiones.
- Pandas: `import pandas as pd` (alias por defecto `pd`).
- Polars: `import polars as pl` (alias habitual `pl`).


In [1]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

pd.__version__, pl.__version__

('2.3.3', '1.34.0')

Datos base que reutilizaremos en todos los ejemplos.
- Propósito: contar con un conjunto sencillo y estable de datos.
- Estructura: columnas `cliente`, `categoria`, `importe`, `fecha`.
- Conversión: `pd.to_datetime` convierte a fecha; en Polars usamos `str.to_date()`.


In [2]:
data = {
    "cliente": ["A", "B", "C", "A", "B", "C"],
    "categoria": ["online", "tienda", "online", "tienda", "online", "tienda"],
    "importe": [120, 80, 200, 150, 90, 50],
    "fecha": ["2024-01-01", "2024-01-02", "2024-01-03", "2024-02-01", "2024-02-02", "2024-02-03"],
}

pdf_base = pd.DataFrame(data)
pdf_base["fecha"] = pd.to_datetime(pdf_base["fecha"])
pldf_base = pl.DataFrame(data).with_columns(pl.col("fecha").str.to_date())

pdf_base.head(), pldf_base.head()

(  cliente categoria  importe      fecha
 0       A    online      120 2024-01-01
 1       B    tienda       80 2024-01-02
 2       C    online      200 2024-01-03
 3       A    tienda      150 2024-02-01
 4       B    online       90 2024-02-02,
 shape: (5, 4)
 ┌─────────┬───────────┬─────────┬────────────┐
 │ cliente ┆ categoria ┆ importe ┆ fecha      │
 │ ---     ┆ ---       ┆ ---     ┆ ---        │
 │ str     ┆ str       ┆ i64     ┆ date       │
 ╞═════════╪═══════════╪═════════╪════════════╡
 │ A       ┆ online    ┆ 120     ┆ 2024-01-01 │
 │ B       ┆ tienda    ┆ 80      ┆ 2024-01-02 │
 │ C       ┆ online    ┆ 200     ┆ 2024-01-03 │
 │ A       ┆ tienda    ┆ 150     ┆ 2024-02-01 │
 │ B       ┆ online    ┆ 90      ┆ 2024-02-02 │
 └─────────┴───────────┴─────────┴────────────┘)

### 1. Creación y estructuras básicas
- Propósito: inicializar `Series` y `DataFrame` listos para análisis.
- Pandas: `pd.Series(data, index=None, dtype=None)` y `pd.DataFrame(data, index=None, columns=None)` (index/columns `None` usa rangos e inferencia por defecto).
- Polars: `pl.Series(name, values, dtype=None)` y `pl.DataFrame(data, schema=None)` (esquema inferido por defecto).


In [3]:
series_pd = pd.Series([1, 2, 3], name="suma")
df_pd = pd.DataFrame({"cliente": ["X", "Y"], "importe": [10, 20]})
series_pl = pl.Series("suma", [1, 2, 3])
df_pl = pl.DataFrame({"cliente": ["X", "Y"], "importe": [10, 20]})
series_pd, df_pd, series_pl, df_pl

(0    1
 1    2
 2    3
 Name: suma, dtype: int64,
   cliente  importe
 0       X       10
 1       Y       20,
 shape: (3,)
 Series: 'suma' [i64]
 [
 	1
 	2
 	3
 ],
 shape: (2, 2)
 ┌─────────┬─────────┐
 │ cliente ┆ importe │
 │ ---     ┆ ---     │
 │ str     ┆ i64     │
 ╞═════════╪═════════╡
 │ X       ┆ 10      │
 │ Y       ┆ 20      │
 └─────────┴─────────┘)

### 2. Importación y exportación de datos
- Propósito: leer/escribir desde texto o disco.
- Pandas: `pd.read_csv(path_or_buffer, sep=",", dtype=None)`; exportar con `df.to_csv(path, index=False)` (index `True` por defecto, solemos poner `False`).
- Polars: `pl.read_csv(source, schema=None)`; exportar con `df.write_csv(path)`.


In [5]:
from io import StringIO
from pathlib import Path

csv_text = """cliente,importe
A,10
B,20
"""
pd_from_csv = pd.read_csv(StringIO(csv_text))
pl_from_csv = pl.read_csv(StringIO(csv_text))

# Demostración de escritura simple y limpieza del archivo de ejemplo
pd_path = Path("demo_pd.csv")
pd_from_csv.to_csv(pd_path, index=False)
pl_path = Path("demo_pl.csv")
pl_from_csv.write_csv(pl_path)

(pd_from_csv.head(), pl_from_csv.head()), pd_path.exists(), pl_path.exists()

((  cliente  importe
  0       A       10
  1       B       20,
  shape: (2, 2)
  ┌─────────┬─────────┐
  │ cliente ┆ importe │
  │ ---     ┆ ---     │
  │ str     ┆ i64     │
  ╞═════════╪═════════╡
  │ A       ┆ 10      │
  │ B       ┆ 20      │
  └─────────┴─────────┘),
 True,
 True)

### 3. Selección y filtrado básico
- Propósito: elegir columnas/filas rápidamente.
- Pandas: `df[[cols]]`, `df.head(n=5)`, `df.loc[filtro, cols]` (`head` por defecto 5 filas).
- Polars: `df.select(cols)`, `df.head(n=5)`, `df.filter(condicion)`.


In [6]:
cols_pd = pdf_base[["cliente", "importe"]]
filtrado_pd = pdf_base.loc[pdf_base["importe"] > 100, ["cliente", "importe"]]
cols_pl = pldf_base.select(["cliente", "importe"])
filtrado_pl = pldf_base.filter(pl.col("importe") > 100).select(["cliente", "importe"])
cols_pd.head(), filtrado_pd.head(), cols_pl.head(), filtrado_pl.head()

(  cliente  importe
 0       A      120
 1       B       80
 2       C      200
 3       A      150
 4       B       90,
   cliente  importe
 0       A      120
 2       C      200
 3       A      150,
 shape: (5, 2)
 ┌─────────┬─────────┐
 │ cliente ┆ importe │
 │ ---     ┆ ---     │
 │ str     ┆ i64     │
 ╞═════════╪═════════╡
 │ A       ┆ 120     │
 │ B       ┆ 80      │
 │ C       ┆ 200     │
 │ A       ┆ 150     │
 │ B       ┆ 90      │
 └─────────┴─────────┘,
 shape: (3, 2)
 ┌─────────┬─────────┐
 │ cliente ┆ importe │
 │ ---     ┆ ---     │
 │ str     ┆ i64     │
 ╞═════════╪═════════╡
 │ A       ┆ 120     │
 │ C       ┆ 200     │
 │ A       ┆ 150     │
 └─────────┴─────────┘)

### 4. Acceso optimizado (escalares)
- Propósito: obtener valores concretos sin construir nuevas series.
- Pandas: `df.at[idx, col]` (acceso por etiqueta), `df.iat[i, j]` (acceso por posición), O(1).
- Polars: `df[col][i]` o `df.row(i)` para fila completa (listas ligeras).


In [7]:
valor_pd_at = pdf_base.at[0, "importe"]
valor_pd_iat = pdf_base.iat[0, 2]
valor_pl_item = pldf_base["importe"][0]
valor_pl_row = pldf_base.row(0)
valor_pd_at, valor_pd_iat, valor_pl_item, valor_pl_row

(np.int64(120),
 np.int64(120),
 120,
 ('A', 'online', 120, datetime.date(2024, 1, 1)))

### 5. Filtrado booleano y consultas
- Propósito: aplicar condiciones complejas de forma declarativa.
- Pandas: `df[df[col] > x]`, `df.query("expr")` (usa nombre de columna en la expresión).
- Polars: `df.filter(pl.col("col") > x)` combinando condiciones con `&`/`|`.


In [8]:
boolean_pd = pdf_base[(pdf_base["categoria"] == "online") & (pdf_base["importe"] > 90)]
query_pd = pdf_base.query("categoria == 'online' and importe > 90")
boolean_pl = pldf_base.filter((pl.col("categoria") == "online") & (pl.col("importe") > 90))
boolean_pd, query_pd, boolean_pl

(  cliente categoria  importe      fecha
 0       A    online      120 2024-01-01
 2       C    online      200 2024-01-03,
   cliente categoria  importe      fecha
 0       A    online      120 2024-01-01
 2       C    online      200 2024-01-03,
 shape: (2, 4)
 ┌─────────┬───────────┬─────────┬────────────┐
 │ cliente ┆ categoria ┆ importe ┆ fecha      │
 │ ---     ┆ ---       ┆ ---     ┆ ---        │
 │ str     ┆ str       ┆ i64     ┆ date       │
 ╞═════════╪═══════════╪═════════╪════════════╡
 │ A       ┆ online    ┆ 120     ┆ 2024-01-01 │
 │ C       ┆ online    ┆ 200     ┆ 2024-01-03 │
 └─────────┴───────────┴─────────┴────────────┘)

### 6. Limpieza y datos faltantes (NaN/Null)
- Propósito: rellenar o descartar valores ausentes.
- Pandas: `df.dropna(subset=None)` (`subset` None afecta todas las columnas), `df.fillna(valor)`.
- Polars: `df.drop_nulls()` (todas las columnas salvo que se indiquen), `df.fill_null(valor)`.


In [10]:
pd_missing = pdf_base.copy()
pd_missing.loc[1, "importe"] = None
pd_limpio = pd_missing.dropna(subset=["importe"])
pd_relleno = pd_missing.fillna({"importe": 0})

pl_missing = pldf_base.with_columns(
    pl.when(pl.col("cliente") == "B").then(None).otherwise(pl.col("importe")).alias("importe")
)
pl_limpio = pl_missing.drop_nulls()
pl_relleno = pl_missing.fill_null(0)

pd_limpio.head(), pd_relleno.head(), pl_limpio.head(), pl_relleno.head()

(  cliente categoria  importe      fecha
 0       A    online    120.0 2024-01-01
 2       C    online    200.0 2024-01-03
 3       A    tienda    150.0 2024-02-01
 4       B    online     90.0 2024-02-02
 5       C    tienda     50.0 2024-02-03,
   cliente categoria  importe      fecha
 0       A    online    120.0 2024-01-01
 1       B    tienda      0.0 2024-01-02
 2       C    online    200.0 2024-01-03
 3       A    tienda    150.0 2024-02-01
 4       B    online     90.0 2024-02-02,
 shape: (4, 4)
 ┌─────────┬───────────┬─────────┬────────────┐
 │ cliente ┆ categoria ┆ importe ┆ fecha      │
 │ ---     ┆ ---       ┆ ---     ┆ ---        │
 │ str     ┆ str       ┆ i64     ┆ date       │
 ╞═════════╪═══════════╪═════════╪════════════╡
 │ A       ┆ online    ┆ 120     ┆ 2024-01-01 │
 │ C       ┆ online    ┆ 200     ┆ 2024-01-03 │
 │ A       ┆ tienda    ┆ 150     ┆ 2024-02-01 │
 │ C       ┆ tienda    ┆ 50      ┆ 2024-02-03 │
 └─────────┴───────────┴─────────┴────────────┘,
 shape: (5

### 7. Reestructuración de índices
- Propósito: definir identificadores lógicos para filas.
- Pandas: `df.set_index(col, drop=True)` (`drop` True elimina columna), `df.reset_index(drop=False)` (`drop` False conserva la columna).
- Polars: no usa índice; se añade columna de índice con `with_row_index(name="idx")`.


In [11]:
pd_idx = pdf_base.set_index("cliente")
pd_reset = pd_idx.reset_index(drop=False)

pl_indexed = pldf_base.with_row_index(name="idx")
pl_indexed.head(), pd_idx.head(), pd_reset.head()

(shape: (5, 5)
 ┌─────┬─────────┬───────────┬─────────┬────────────┐
 │ idx ┆ cliente ┆ categoria ┆ importe ┆ fecha      │
 │ --- ┆ ---     ┆ ---       ┆ ---     ┆ ---        │
 │ u32 ┆ str     ┆ str       ┆ i64     ┆ date       │
 ╞═════╪═════════╪═══════════╪═════════╪════════════╡
 │ 0   ┆ A       ┆ online    ┆ 120     ┆ 2024-01-01 │
 │ 1   ┆ B       ┆ tienda    ┆ 80      ┆ 2024-01-02 │
 │ 2   ┆ C       ┆ online    ┆ 200     ┆ 2024-01-03 │
 │ 3   ┆ A       ┆ tienda    ┆ 150     ┆ 2024-02-01 │
 │ 4   ┆ B       ┆ online    ┆ 90      ┆ 2024-02-02 │
 └─────┴─────────┴───────────┴─────────┴────────────┘,
         categoria  importe      fecha
 cliente                              
 A          online      120 2024-01-01
 B          tienda       80 2024-01-02
 C          online      200 2024-01-03
 A          tienda      150 2024-02-01
 B          online       90 2024-02-02,
   cliente categoria  importe      fecha
 0       A    online      120 2024-01-01
 1       B    tienda       80 2024

### 8. Reestructuración y pivoteo
- Propósito: pasar de formato largo a ancho o viceversa.
- Pandas: `df.pivot_table(values, index, columns, aggfunc="mean", fill_value=None)` (`aggfunc` por defecto `mean`).
- Polars: `df.pivot(values, index, columns, aggregate_function="first")` (hay que indicar función de agregación, p.e. `"sum"`).


In [12]:
pivot_pd = pdf_base.pivot_table(
    values="importe", index="cliente", columns="categoria", aggfunc="sum", fill_value=0
)
pivot_pl = pldf_base.pivot(values="importe", index="cliente", columns="categoria", aggregate_function="sum")
pivot_pd, pivot_pl

/tmp/ipykernel_30655/3023205863.py:4: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  pivot_pl = pldf_base.pivot(values="importe", index="cliente", columns="categoria", aggregate_function="sum")


(categoria  online  tienda
 cliente                  
 A             120     150
 B              90      80
 C             200      50,
 shape: (3, 3)
 ┌─────────┬────────┬────────┐
 │ cliente ┆ online ┆ tienda │
 │ ---     ┆ ---    ┆ ---    │
 │ str     ┆ i64    ┆ i64    │
 ╞═════════╪════════╪════════╡
 │ A       ┆ 120    ┆ 150    │
 │ B       ┆ 90     ┆ 80     │
 │ C       ┆ 200    ┆ 50     │
 └─────────┴────────┴────────┘)

### 9. Manipulación de datos duplicados
- Propósito: detectar y eliminar repeticiones.
- Pandas: `df.duplicated(subset=None, keep="first")` (`keep` controla qué duplicado marcar), `df.drop_duplicates(subset=None)`.
- Polars: `df.is_duplicated(subset=None)`, `df.unique(subset=None, keep="first")`.


In [ ]:
pd_dups = pd.concat([pdf_base, pdf_base.iloc[[0]]], ignore_index=True)
pd_unique = pd_dups.drop_duplicates(subset=["cliente", "fecha"])

pl_dups = pldf_base.vstack(pldf_base.head(1))
pl_unique = pl_dups.unique(subset=["cliente", "fecha"])
pd_dups.duplicated(subset=["cliente", "fecha"]).head(), pd_unique.head(), pl_unique.head()

### 10. Combinación y fusión de datos
- Propósito: enriquecer datos con tablas auxiliares.
- Pandas: `df.merge(otra, on, how="inner")` (`how` por defecto `inner`).
- Polars: `df.join(otra, on, how="inner")` (`how` por defecto `inner`).


In [ ]:
clientes_info_pd = pd.DataFrame({"cliente": ["A", "B", "C"], "segmento": ["oro", "plata", "bronce"]})
pd_merge = pdf_base.merge(clientes_info_pd, on="cliente", how="left")

clientes_info_pl = pl.DataFrame({"cliente": ["A", "B", "C"], "segmento": ["oro", "plata", "bronce"]})
pl_merge = pldf_base.join(clientes_info_pl, on="cliente", how="left")
pd_merge.head(), pl_merge.head()

### 11. Estadísticas descriptivas
- Propósito: obtener resumen rápido de distribución.
- Pandas: `df.describe(percentiles=None)` (`include`/`exclude` opcionales), por defecto percentiles 0.25, 0.5, 0.75.
- Polars: `df.describe()` resume todas las columnas numéricas; también `df.select(pl.col("col").mean())` para métricas concretas.


In [ ]:
pd_desc = pdf_base["importe"].describe()
pl_desc = pldf_base.select(pl.col("importe").describe())
pd_desc, pl_desc

### 12. Agregación y agrupación (split-apply-combine)
- Propósito: resumir por grupos.
- Pandas: `df.groupby(keys)[col].agg(funcs)` (`as_index` True por defecto).
- Polars: `df.group_by(keys).agg(expresiones)`.


In [ ]:
pd_group = pdf_base.groupby("cliente")["importe"].agg(["count", "mean", "sum"])
pl_group = pldf_base.group_by("cliente").agg([
    pl.len().alias("count"),
    pl.mean("importe").alias("mean"),
    pl.sum("importe").alias("sum"),
])
pd_group, pl_group

### 13. Aplicación y transformación de grupos
- Propósito: calcular métricas por grupo y reutilizarlas a nivel de fila.
- Pandas: `groupby().transform(func)` conserva el tamaño original.
- Polars: `with_columns(expr.over("grupo"))` aplica la ventana sobre cada grupo.


In [ ]:
pd_group_pct = pdf_base.copy()
pd_group_pct["pct_cliente"] = pd_group_pct.groupby("cliente")["importe"].transform(lambda s: s / s.sum())

pl_group_pct = pldf_base.with_columns(
    (pl.col("importe") / pl.col("importe").sum().over("cliente")).alias("pct_cliente")
)
pd_group_pct[["cliente", "importe", "pct_cliente"]].head(), pl_group_pct.select(["cliente", "importe", "pct_cliente"]).head()

### 14. Modificación condicional
- Propósito: crear columnas según reglas lógicas.
- Pandas: `np.where(cond, valor_si, valor_no)` o `df.assign(nueva=...)`.
- Polars: `pl.when(cond).then(valor).otherwise(valor)` dentro de `with_columns`.


In [ ]:
pd_cond = pdf_base.copy()
pd_cond["nivel"] = np.where(pd_cond["importe"] > 100, "alto", "medio")

pl_cond = pldf_base.with_columns(
    pl.when(pl.col("importe") > 100).then("alto").otherwise("medio").alias("nivel")
)
pd_cond[["cliente", "importe", "nivel"]].head(), pl_cond.head()

### 15. Conversión de tipos (type casting)
- Propósito: asegurar tipos correctos para cálculos o E/S.
- Pandas: `df[col].astype(nuevo_tipo, errors="raise")` (por defecto lanza error si falla).
- Polars: `pl.col("col").cast(tipo, strict=True)` (`strict` True lanza error si no puede convertir).


In [ ]:
pd_cast = pdf_base.assign(importe_texto=pdf_base["importe"].astype(str))
pl_cast = pldf_base.with_columns(pl.col("importe").cast(pl.Float64))
pd_cast.dtypes, pl_cast.dtypes

### 16. Manipulación de texto (vectorizado)
- Propósito: limpiar y transformar cadenas.
- Pandas: `df[col].str.lower()`, `str.replace(pat, repl, regex=True)`.
- Polars: `pl.col("col").str.to_lowercase()`, `str.replace_all(pat, repl)`.


In [ ]:
pd_text = pdf_base.copy()
pd_text["cliente_min"] = pd_text["cliente"].str.lower()

pl_text = pldf_base.with_columns(pl.col("cliente").str.to_lowercase().alias("cliente_min"))
pd_text[["cliente", "cliente_min"]].head(), pl_text.select(["cliente", "cliente_min"]).head()

### 17. Series de tiempo y fechas
- Propósito: extraer componentes temporales y reamostrar.
- Pandas: `pd.to_datetime`, `df.resample(freq).agg(...)` (requiere índice de fecha).
- Polars: `pl.col("fecha").dt.*` (p.e. `dt.month()`), y `group_by_dynamic` para ventanas.


In [ ]:
pd_fecha = pdf_base.copy()
pd_fecha["mes"] = pd_fecha["fecha"].dt.to_period("M")

pl_fecha = pldf_base.with_columns(pl.col("fecha").dt.month().alias("mes"))

pd_agg_mes = pd_fecha.groupby("mes")["importe"].sum()
pl_agg_mes = pl_fecha.group_by("mes").agg(pl.sum("importe"))
pd_agg_mes, pl_agg_mes

### 18. Discretización y cuantificación
- Propósito: agrupar valores continuos en tramos.
- Pandas: `pd.cut(x, bins, labels=None, include_lowest=False)` (etiquetas opcionales).
- Polars: no tiene `cut` idéntico; se acostumbra usar `when/then` encadenado o `qcut` vía pandas.


In [ ]:
pd_bins = pdf_base.copy()
pd_bins["tramo"] = pd.cut(pd_bins["importe"], bins=[0, 80, 150, 300], labels=["bajo", "medio", "alto"])

pl_bins = pldf_base.with_columns(
    pl.when(pl.col("importe") <= 80)
    .then("bajo")
    .when(pl.col("importe") <= 150)
    .then("medio")
    .otherwise("alto")
    .alias("tramo")
)
pd_bins[["importe", "tramo"]].head(), pl_bins.select(["importe", "tramo"]).head()

### 19. Visualización rápida (plotting)
- Propósito: obtener gráficos exploratorios rápidos.
- Pandas: `df.plot(kind="bar", x=..., y=...)` (usa Matplotlib por defecto).
- Polars: delega en otras librerías; se suele convertir a pandas con `df.to_pandas().plot(...)`.


In [ ]:
ax1 = pdf_base.plot(kind="bar", x="cliente", y="importe", title="Pandas plot")
plt.tight_layout()

pldf_base.to_pandas().plot(kind="bar", x="cliente", y="importe", title="Polars via pandas")
plt.tight_layout()

### 20. Funciones de trabajo y eficiencia
- Propósito: procesar más rápido y con menos memoria.
- Pandas: `pd.read_csv(chunksize=N)` para procesar por bloques, `df.eval/df.assign` para expresiones más rápidas.
- Polars: modo perezoso `df.lazy()` y `pl.scan_csv` permiten optimizar y ejecutar al final con `collect()`.


In [ ]:
# Pandas por bloques
chunk_suma = sum(
    chunk["importe"].sum()
    for chunk in pd.read_csv(Path("demo_pd.csv"), chunksize=2)
)

# Polars en modo perezoso
pl_lazy = pldf_base.lazy().group_by("cliente").agg(pl.sum("importe").alias("total"))
pl_lazy_result = pl_lazy.collect()
chunk_suma, pl_lazy_result

### 21. Ventanas y cálculos móviles (extra)
- Propósito: estadísticas sobre ventanas deslizantes.
- Pandas: `df.set_index(fecha)[col].rolling(window).mean()` (`window` suele ser entero o periodo como `"7D"`).
- Polars: `pl.col("col").rolling_mean(window_size)` después de ordenar.


In [ ]:
pd_roll = (
    pdf_base.sort_values("fecha")
    .set_index("fecha")["importe"]
    .rolling(window=2)
    .mean()
)

pl_roll = (
    pldf_base.sort("fecha")
    .with_columns(pl.col("importe").rolling_mean(window_size=2).alias("media_2"))
    .select(["fecha", "media_2"])
)
pd_roll.head(), pl_roll.head()

Limpieza final de archivos temporales usados en los ejemplos de I/O.

In [ ]:
for path in [Path("demo_pd.csv"), Path("demo_pl.csv")]:
    if path.exists():
        path.unlink()
[Path("demo_pd.csv").exists(), Path("demo_pl.csv").exists()]